# CIFAR-10 Image Classification using ANN and CNN

This notebook compares ANN and CNN architectures on the CIFAR-10 dataset and explores training improvements such as data augmentation and EarlyStopping.

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    Flatten,
    Conv2D,
    MaxPooling2D,
    BatchNormalization,
    RandomFlip,
    RandomRotation,
    RandomZoom
)

from tensorflow.keras.callbacks import EarlyStopping

## Dataset Loading

The CIFAR-10 dataset contains 60,000 color images belonging to 10 classes.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Smaller subset for faster training
x_train = x_train[:15000]
y_train = y_train[:15000]

x_test = x_test[:3000]
y_test = y_test[:3000]

print("Train Shape:", x_train.shape)
print("Test Shape:", x_test.shape)

## Sample Images

A few examples from the dataset are displayed below.

In [ ]:
plt.figure(figsize=(10,5))
for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis('off')
plt.tight_layout()
plt.show()

## Data Preprocessing

Pixel values are normalized to improve training stability.

In [ ]:
x_train_norm = x_train / 255.0
x_test_norm = x_test / 255.0

x_train_flat = x_train_norm.reshape(len(x_train_norm), -1)
x_test_flat = x_test_norm.reshape(len(x_test_norm), -1)

## ANN Model

The ANN receives flattened image vectors as input.

In [ ]:
ann_model = models.Sequential([
    layers.Input(shape=(3072,)),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dense(10, activation='softmax')
])

ann_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

ann_history = ann_model.fit(
    x_train_flat, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1
)

ann_test_loss, ann_test_acc = ann_model.evaluate(x_test_flat, y_test)
print("ANN Accuracy:", ann_test_acc)

## CNN Model

The CNN uses convolution and pooling layers to preserve spatial information.

In [ ]:
cnn_model = models.Sequential([
    layers.Conv2D(32,(3,3),activation='relu',input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64,(3,3),activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128,(3,3),activation='relu'),
    layers.Flatten(),
    layers.Dense(128,activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10,activation='softmax')
])

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_history = cnn_model.fit(
    x_train_norm, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1
)

cnn_test_loss, cnn_test_acc = cnn_model.evaluate(x_test_norm, y_test)
print("CNN Accuracy:", cnn_test_acc)

## Validation Accuracy Comparison

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(ann_history.history['val_accuracy'], label='ANN')
plt.plot(cnn_history.history['val_accuracy'], label='CNN')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.title('ANN vs CNN Validation Accuracy')
plt.legend()
plt.show()

## Data Augmentation Model

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

aug_model = models.Sequential([
    data_augmentation,
    layers.Conv2D(32,3,activation='relu',input_shape=(32,32,3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64,3,activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128,activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10,activation='softmax')
])

aug_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

aug_history = aug_model.fit(
    x_train_norm, y_train,
    epochs=10,
    validation_split=0.1
)

aug_loss, aug_acc = aug_model.evaluate(x_test_norm, y_test)
print("Augmented CNN Accuracy:", aug_acc)

## Additional Experiment 1 - Deeper ANN

In [ ]:
deep_ann = models.Sequential([
    layers.Input(shape=(3072,)),
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(512, activation='relu'),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

## Additional Experiment 2 - Larger CNN Filters

In [ ]:
improved_cnn = models.Sequential([
    layers.Conv2D(64,3,activation='relu',input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128,3,activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(256,3,activation='relu'),
    layers.Flatten(),
    layers.Dense(256,activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10,activation='softmax')
])

## Additional Experiment 3 and 4 - 20 Epochs with EarlyStopping

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

improved_cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

improved_history = improved_cnn.fit(
    x_train_norm, y_train,
    epochs=20,
    validation_split=0.1,
    callbacks=[early_stop]
)

improved_loss, improved_acc = improved_cnn.evaluate(x_test_norm, y_test)
print("Improved CNN Accuracy:", improved_acc)

## Final Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Model':['ANN','CNN','Augmented CNN','Improved CNN'],
    'Test Accuracy':[ann_test_acc, cnn_test_acc, aug_acc, improved_acc]
})

comparison

## Conclusion

CNN models generally outperform ANN models because they preserve spatial information. Data augmentation, deeper architectures, larger filter sizes, and EarlyStopping can further improve performance and generalization.